In [13]:
import sys

print("Python do notebook:")
print(sys.executable)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

print("\nReportLab OK")

Python do notebook:
C:\Users\beelt\Documents\collections_case_candidate\.venv\Scripts\python.exe

ReportLab OK


In [14]:
from pathlib import Path
import json
import io
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak,
    Image,
)

warnings.filterwarnings("ignore")

## 1. Configuração

Por padrão, o gerador procura os notebooks na mesma pasta deste arquivo ou na pasta anterior.  
Ajuste apenas os caminhos se a estrutura do repositório mudar.

In [15]:
ROOT = Path.cwd()

CANDIDATES = [
    ROOT / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT.parent / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT / "notebooks" / "05_bivariate_eda_collections_macro_analysis.ipynb",
]

BIVARIATE_NOTEBOOK = next((p for p in CANDIDATES if p.exists()), None)

if BIVARIATE_NOTEBOOK is None:
    raise FileNotFoundError(
        "Não encontrei 05_bivariate_eda_collections_macro_analysis.ipynb. "
        "Ajuste BIVARIATE_NOTEBOOK nesta célula."
    )

REPORT_DIR = ROOT / "reports"
CHART_DIR = REPORT_DIR / "collections_report_charts"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = REPORT_DIR / "collections_macro_bivariate_analysis_report.pdf"

print("Notebook fonte :", BIVARIATE_NOTEBOOK.resolve())
print("PDF de saída   :", PDF_PATH.resolve())

Notebook fonte : C:\Users\beelt\Documents\collections_case_candidate\notebooks\05_bivariate_eda_collections_macro_analysis.ipynb
PDF de saída   : C:\Users\beelt\Documents\collections_case_candidate\notebooks\reports\collections_macro_bivariate_analysis_report.pdf


In [16]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")
QUEUE_PATH = DATA_DIR / "raw" / "collections_queue_sep2026.csv"
WA_PATH = DATA_DIR / "raw" / "whatsapp_collections_history.csv"

if not QUEUE_PATH.exists():
    QUEUE_PATH = Path("/mnt/data/collections_queue_sep2026(1).csv")
if not WA_PATH.exists():
    WA_PATH = Path("/mnt/data/whatsapp_collections_history(2).csv")

queue = pd.read_csv(QUEUE_PATH)
wa = pd.read_csv(WA_PATH)

wa["sent_at"] = pd.to_datetime(wa["sent_at"], errors="coerce")
queue["in_collections_since"] = pd.to_datetime(queue["in_collections_since"], errors="coerce")

print("Queue:", queue.shape)
print("WhatsApp:", wa.shape)

Queue: (10658, 10)
WhatsApp: (75406, 17)


## _______________________________________________________________ "" ____________________________________________________________

In [17]:
# ============================================================
# TEST 1 — EARLY CADENCE vs EVENTUAL FULL SETTLEMENT
# ============================================================
#
# BUSINESS QUESTION
# Can stronger messaging pressure early in the collections
# journey accelerate settlement and reduce the total number
# of messages required per customer?
#
# EXPOSURE:
# Number of WhatsApp messages during DPD 1–7
#
# OUTCOMES:
# - Full settlement
# - Any payment after DPD 7
# - Messages after DPD 7
# - Total messages in observed journey
# - Total recovered amount
# - Maximum observed DPD
#
# IMPORTANT:
# We only include customers observed near the beginning
# of their collections journey (first observed DPD <= 3).
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 0. PREPARE DATA
# ------------------------------------------------------------

df = wa.copy()

df["sent_at"] = pd.to_datetime(df["sent_at"])

df = (
    df.sort_values(
        ["customer_id", "sent_at"]
    )
    .reset_index(drop=True)
)

print("=" * 80)
print("TEST 1 — EARLY CADENCE vs EVENTUAL FULL SETTLEMENT")
print("=" * 80)

print(f"Rows             : {len(df):,}")
print(f"Customers        : {df['customer_id'].nunique():,}")
print(
    f"Observed period  : "
    f"{df['sent_at'].min().date()} → "
    f"{df['sent_at'].max().date()}"
)


# ------------------------------------------------------------
# 1. IDENTIFY CUSTOMERS OBSERVED FROM EARLY JOURNEY
# ------------------------------------------------------------
#
# We don't want customers first appearing at DPD 20, 30, etc.
# because we don't observe their true early cadence.
#
# Keep customers whose first observed DPD <= 3.
# ------------------------------------------------------------

journey = (
    df.groupby("customer_id")
    .agg(
        first_dpd=("days_past_due", "min"),
        max_dpd=("days_past_due", "max"),
        first_contact_at=("sent_at", "min"),
        last_contact_at=("sent_at", "max"),
        total_messages_observed=("message_id", "count")
    )
    .reset_index()
)

eligible_ids = journey.loc[
    journey["first_dpd"] <= 3,
    "customer_id"
]

x = (
    df.loc[
        df["customer_id"].isin(eligible_ids)
    ]
    .copy()
)

print("\n" + "=" * 80)
print("ELIGIBLE POPULATION")
print("=" * 80)

print(
    f"Customers observed overall       : "
    f"{df['customer_id'].nunique():,}"
)

print(
    f"Customers with first DPD <= 3    : "
    f"{x['customer_id'].nunique():,}"
)

print(
    f"Population retained              : "
    f"{x['customer_id'].nunique() / df['customer_id'].nunique():.1%}"
)


# ------------------------------------------------------------
# 2. EARLY EXPOSURE WINDOW
# ------------------------------------------------------------
#
# Early journey = DPD 1–7
#
# Measure:
# - number of messages
# - number of distinct contact days
# ------------------------------------------------------------

early = (
    x.loc[
        x["days_past_due"].between(1, 7)
    ]
    .copy()
)

early_customer = (
    early.groupby("customer_id")
    .agg(
        early_messages=(
            "message_id",
            "count"
        ),

        early_contact_days=(
            "sent_at",
            lambda s: s.dt.date.nunique()
        ),

        first_early_message_at=(
            "sent_at",
            "min"
        ),

        last_early_message_at=(
            "sent_at",
            "max"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 3. CADENCE BUCKETS
# ------------------------------------------------------------
#
# Initial descriptive segmentation:
#
# Low    = 1–2 messages
# Medium = 3–4 messages
# High   = 5+ messages
# ------------------------------------------------------------

early_customer["early_cadence"] = pd.cut(
    early_customer["early_messages"],
    bins=[0, 2, 4, np.inf],
    labels=[
        "Low: 1-2",
        "Medium: 3-4",
        "High: 5+"
    ],
    right=True
)


# ------------------------------------------------------------
# 4. POST-EARLY-WINDOW OUTCOMES
# ------------------------------------------------------------
#
# IMPORTANT:
# Exposure is measured during DPD 1–7.
#
# Here we measure what happens AFTER DPD 7.
# ------------------------------------------------------------

post = (
    x.loc[
        x["days_past_due"] > 7
    ]
    .copy()
)

post_customer = (
    post.groupby("customer_id")
    .agg(
        post_messages=(
            "message_id",
            "count"
        ),

        post_recovered_brl=(
            "amount_paid_brl",
            "sum"
        ),

        post_any_payment=(
            "amount_paid_brl",
            lambda s: (s.fillna(0) > 0).any()
        ),

        post_max_dpd=(
            "days_past_due",
            "max"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 5. FULL OBSERVED JOURNEY
# ------------------------------------------------------------

full_customer = (
    x.groupby("customer_id")
    .agg(
        total_messages=(
            "message_id",
            "count"
        ),

        total_recovered_brl=(
            "amount_paid_brl",
            "sum"
        ),

        initial_balance_brl=(
            "outstanding_balance_brl",
            "first"
        ),

        last_balance_brl=(
            "outstanding_balance_brl",
            "last"
        ),

        max_dpd=(
            "days_past_due",
            "max"
        ),

        first_message_at=(
            "sent_at",
            "min"
        ),

        last_message_at=(
            "sent_at",
            "max"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 6. FULL SETTLEMENT FLAG
# ------------------------------------------------------------
#
# Conservative observed proxy:
# last observed outstanding balance <= R$ 0.05
#
# IMPORTANT:
# If your canonical settlement definition already exists
# elsewhere, we can replace this flag later.
# ------------------------------------------------------------

TOLERANCE_BRL = 0.05

full_customer["fully_settled"] = (
    full_customer["last_balance_brl"]
    <= TOLERANCE_BRL
)


# ------------------------------------------------------------
# 7. REACHED DPD 30 / 60
# ------------------------------------------------------------

full_customer["reached_dpd30"] = (
    full_customer["max_dpd"] >= 30
)

full_customer["reached_dpd60"] = (
    full_customer["max_dpd"] >= 60
)


# ------------------------------------------------------------
# 8. COMBINE CUSTOMER-LEVEL TABLES
# ------------------------------------------------------------

analysis = (
    early_customer
    .merge(
        post_customer,
        on="customer_id",
        how="left"
    )
    .merge(
        full_customer,
        on="customer_id",
        how="left"
    )
)


# ------------------------------------------------------------
# 9. FILL CUSTOMERS WITH NO POST-DPD7 ACTIVITY
# ------------------------------------------------------------

analysis["post_messages"] = (
    analysis["post_messages"]
    .fillna(0)
    .astype(int)
)

analysis["post_recovered_brl"] = (
    analysis["post_recovered_brl"]
    .fillna(0)
)

analysis["post_any_payment"] = (
    analysis["post_any_payment"]
    .fillna(False)
)

analysis["reached_dpd30"] = (
    analysis["reached_dpd30"]
    .fillna(False)
)

analysis["reached_dpd60"] = (
    analysis["reached_dpd60"]
    .fillna(False)
)


# ------------------------------------------------------------
# 10. MESSAGES AFTER EARLY WINDOW
# ------------------------------------------------------------
#
# This is one of our most important operational metrics.
#
# If stronger early cadence works, ideally:
#
# early messages ↑
# BUT
# later messages ↓
# AND
# total messages ↓ or stay controlled
# AND
# settlement ↑
# ------------------------------------------------------------

analysis["share_messages_early"] = (
    analysis["early_messages"]
    / analysis["total_messages"]
)


# ------------------------------------------------------------
# 11. JOURNEY DURATION
# ------------------------------------------------------------

analysis["observed_journey_days"] = (
    analysis["last_message_at"]
    - analysis["first_message_at"]
).dt.total_seconds() / 86400


# ------------------------------------------------------------
# 12. MAIN CADENCE SUMMARY
# ------------------------------------------------------------

cadence_summary = (
    analysis
    .groupby(
        "early_cadence",
        observed=True
    )
    .agg(

        # Population
        customers=(
            "customer_id",
            "nunique"
        ),

        # Early pressure
        avg_early_messages=(
            "early_messages",
            "mean"
        ),

        median_early_messages=(
            "early_messages",
            "median"
        ),

        avg_early_contact_days=(
            "early_contact_days",
            "mean"
        ),

        # Settlement
        fully_settled_rate=(
            "fully_settled",
            "mean"
        ),

        # Payment after early window
        post_payment_rate=(
            "post_any_payment",
            "mean"
        ),

        # Messages AFTER DPD 7
        avg_post_messages=(
            "post_messages",
            "mean"
        ),

        median_post_messages=(
            "post_messages",
            "median"
        ),

        # Total messaging burden
        avg_total_messages=(
            "total_messages",
            "mean"
        ),

        median_total_messages=(
            "total_messages",
            "median"
        ),

        # Recovery
        avg_recovered_brl=(
            "total_recovered_brl",
            "mean"
        ),

        median_recovered_brl=(
            "total_recovered_brl",
            "median"
        ),

        # DPD
        avg_max_dpd=(
            "max_dpd",
            "mean"
        ),

        reached_dpd30_rate=(
            "reached_dpd30",
            "mean"
        ),

        reached_dpd60_rate=(
            "reached_dpd60",
            "mean"
        ),

        # Journey length
        avg_journey_days=(
            "observed_journey_days",
            "mean"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 13. CONVERT RATES TO %
# ------------------------------------------------------------

rate_columns = [
    "fully_settled_rate",
    "post_payment_rate",
    "reached_dpd30_rate",
    "reached_dpd60_rate"
]

for col in rate_columns:
    cadence_summary[col] *= 100


# ------------------------------------------------------------
# 14. ROUND OUTPUT
# ------------------------------------------------------------

numeric_cols = cadence_summary.select_dtypes(
    include="number"
).columns

cadence_summary[numeric_cols] = (
    cadence_summary[numeric_cols]
    .round(2)
)


# ------------------------------------------------------------
# 15. DISPLAY MAIN RESULT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("EARLY CADENCE × COLLECTIONS OUTCOME")
print("=" * 80)

display(cadence_summary)


# ------------------------------------------------------------
# 16. MORE GRANULAR VIEW
# ------------------------------------------------------------
#
# Don't rely only on arbitrary Low / Medium / High buckets.
#
# Show exact number of messages during DPD 1–7.
# ------------------------------------------------------------

exact_cadence = (
    analysis
    .groupby(
        "early_messages",
        observed=True
    )
    .agg(

        customers=(
            "customer_id",
            "nunique"
        ),

        fully_settled_rate=(
            "fully_settled",
            "mean"
        ),

        post_payment_rate=(
            "post_any_payment",
            "mean"
        ),

        avg_post_messages=(
            "post_messages",
            "mean"
        ),

        avg_total_messages=(
            "total_messages",
            "mean"
        ),

        median_total_messages=(
            "total_messages",
            "median"
        ),

        avg_recovered_brl=(
            "total_recovered_brl",
            "mean"
        ),

        reached_dpd30_rate=(
            "reached_dpd30",
            "mean"
        ),

        reached_dpd60_rate=(
            "reached_dpd60",
            "mean"
        )
    )
    .reset_index()
)


for col in [
    "fully_settled_rate",
    "post_payment_rate",
    "reached_dpd30_rate",
    "reached_dpd60_rate"
]:
    exact_cadence[col] *= 100


numeric_cols = exact_cadence.select_dtypes(
    include="number"
).columns

exact_cadence[numeric_cols] = (
    exact_cadence[numeric_cols]
    .round(2)
)


print("\n" + "=" * 80)
print("EXACT NUMBER OF EARLY MESSAGES")
print("=" * 80)

display(exact_cadence)


# ------------------------------------------------------------
# 17. CUSTOMER-LEVEL ANALYTICAL BASE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ANALYTICAL BASE")
print("=" * 80)

print(f"Customers analysed : {len(analysis):,}")

display(
    analysis[
        [
            "customer_id",
            "early_cadence",
            "early_messages",
            "early_contact_days",
            "fully_settled",
            "post_any_payment",
            "post_messages",
            "total_messages",
            "total_recovered_brl",
            "initial_balance_brl",
            "last_balance_brl",
            "max_dpd",
            "reached_dpd30",
            "reached_dpd60",
            "observed_journey_days"
        ]
    ].head(20)
)

TEST 1 — EARLY CADENCE vs EVENTUAL FULL SETTLEMENT
Rows             : 75,406
Customers        : 11,724
Observed period  : 2026-06-01 → 2026-08-31

ELIGIBLE POPULATION
Customers observed overall       : 11,724
Customers with first DPD <= 3    : 8,068
Population retained              : 68.8%

EARLY CADENCE × COLLECTIONS OUTCOME


,early_cadence,customers,avg_early_messages,median_early_messages,avg_early_contact_days,fully_settled_rate,post_payment_rate,avg_post_messages,median_post_messages,avg_total_messages,median_total_messages,avg_recovered_brl,median_recovered_brl,avg_max_dpd,reached_dpd30_rate,reached_dpd60_rate,avg_journey_days
0,Low: 1-2,4710,1.60,2.00,1.60,0.00,24.63,3.80,3.00,5.40,5.00,333.69,0.00,23.68,36.58,1.59,21.84
1,Medium: 3-4,3093,3.29,3.00,3.29,0.00,27.51,4.89,5.00,8.18,8.00,272.13,0.00,30.13,47.79,2.52,28.50
2,High: 5+,265,5.11,5.00,5.11,0.00,25.66,5.04,5.00,10.15,10.00,206.13,0.00,31.47,51.32,2.64,30.09



EXACT NUMBER OF EARLY MESSAGES


,early_messages,customers,fully_settled_rate,post_payment_rate,avg_post_messages,avg_total_messages,median_total_messages,avg_recovered_brl,reached_dpd30_rate,reached_dpd60_rate
0,1,1872,0.00,20.35,2.97,3.97,3.00,385.10,28.10,1.34
1,2,2838,0.00,27.45,4.34,6.34,6.00,299.77,42.18,1.76
2,3,2194,0.00,28.17,4.74,7.74,8.00,280.60,46.31,2.55
3,4,899,0.00,25.92,5.25,9.25,9.00,251.45,51.39,2.45
4,5,236,0.00,25.42,5.01,10.01,10.00,209.59,51.27,2.97
5,6,28,0.00,28.57,5.21,11.21,12.00,184.29,50.00,0.00
6,7,1,0.00,0.00,8.00,15.00,15.00,0.00,100.00,0.00



ANALYTICAL BASE
Customers analysed : 8,068


,customer_id,early_cadence,early_messages,early_contact_days,fully_settled,post_any_payment,post_messages,total_messages,total_recovered_brl,initial_balance_brl,last_balance_brl,max_dpd,reached_dpd30,reached_dpd60,observed_journey_days
0,C000001,Medium: 3-4,3,3,False,True,4,7,934.58,934.58,934.58,26,False,False,22.82
1,C000003,Medium: 3-4,4,4,False,False,8,12,0.00,758.22,758.22,60,True,True,57.04
2,C000004,Medium: 3-4,3,3,False,True,2,5,"1,331.09","1,331.09","1,331.09",11,False,False,9.72
3,C000005,Medium: 3-4,3,3,False,False,2,5,0.00,250.35,250.35,21,False,False,20.19
4,C000006,Medium: 3-4,4,4,False,True,7,11,415.00,488.23,488.23,34,True,False,32.02
5,C000007,Low: 1-2,2,2,False,False,9,11,0.00,314.44,314.44,49,True,False,48.05
6,C000008,Medium: 3-4,4,4,False,False,8,12,0.00,927.60,927.60,53,True,False,51.71
7,C000009,Medium: 3-4,3,3,False,True,4,7,878.15,878.15,878.15,56,True,False,54.81
8,C000011,Low: 1-2,1,1,False,True,10,11,456.62,966.15,509.53,48,True,False,46.96
9,C000012,Medium: 3-4,3,3,False,True,1,4,320.23,320.23,320.23,18,False,False,16.67


In [18]:
# ============================================================
# TEST 1 — EARLY CADENCE vs EVENTUAL FULL SETTLEMENT
# ============================================================
#
# BUSINESS QUESTION
#
# Can stronger messaging pressure early in the collections
# journey increase eventual settlement and reduce the total
# number of messages required per customer?
#
# ------------------------------------------------------------
# EXPOSURE
# ------------------------------------------------------------
#
# Number of WhatsApp messages during DPD 1–7.
#
# ------------------------------------------------------------
# ELIGIBLE POPULATION
# ------------------------------------------------------------
#
# Customers whose first observed DPD <= 3.
#
# This avoids analysing customers whose early collections
# journey was not observed.
#
# ------------------------------------------------------------
# CANONICAL FULL SETTLEMENT
# ------------------------------------------------------------
#
# WITHOUT discount_offer:
#
#   total recovered >= initial observed balance
#
#
# WITH discount_offer:
#
#   required recovery =
#
#       payments already made BEFORE first discount
#       +
#       85% × balance observed at first discount
#
#
# IMPORTANT:
#
# We DO NOT use last outstanding balance == 0.
#
# ============================================================


import pandas as pd
import numpy as np


# ============================================================
# PARAMETERS
# ============================================================

TOLERANCE_BRL = 0.05

DISCOUNT_SETTLEMENT_RATE = 0.85

EARLY_DPD_MIN = 1
EARLY_DPD_MAX = 7

MAX_FIRST_DPD = 3


# ============================================================
# 0. PREPARE DATA
# ============================================================

df = wa.copy()

df["sent_at"] = pd.to_datetime(
    df["sent_at"]
)

df["amount_paid_brl"] = (
    pd.to_numeric(
        df["amount_paid_brl"],
        errors="coerce"
    )
    .fillna(0)
)

df["outstanding_balance_brl"] = (
    pd.to_numeric(
        df["outstanding_balance_brl"],
        errors="coerce"
    )
)

df["days_past_due"] = (
    pd.to_numeric(
        df["days_past_due"],
        errors="coerce"
    )
)

df = (
    df.sort_values(
        [
            "customer_id",
            "sent_at"
        ]
    )
    .reset_index(drop=True)
)


print("=" * 90)
print("TEST 1 — EARLY CADENCE vs EVENTUAL FULL SETTLEMENT")
print("=" * 90)

print(
    f"Rows                         : "
    f"{len(df):,}"
)

print(
    f"Customers                    : "
    f"{df['customer_id'].nunique():,}"
)

print(
    f"Observed period              : "
    f"{df['sent_at'].min().date()} "
    f"→ "
    f"{df['sent_at'].max().date()}"
)


# ============================================================
# 1. IDENTIFY CUSTOMERS OBSERVED FROM EARLY JOURNEY
# ============================================================

journey = (
    df.groupby("customer_id")
    .agg(

        first_dpd=(
            "days_past_due",
            "min"
        ),

        max_dpd=(
            "days_past_due",
            "max"
        ),

        first_contact_at=(
            "sent_at",
            "min"
        ),

        last_contact_at=(
            "sent_at",
            "max"
        ),

        total_messages_observed=(
            "message_id",
            "count"
        )
    )
    .reset_index()
)


eligible_ids = (
    journey.loc[
        journey["first_dpd"] <= MAX_FIRST_DPD,
        "customer_id"
    ]
)


x = (
    df.loc[
        df["customer_id"].isin(
            eligible_ids
        )
    ]
    .sort_values(
        [
            "customer_id",
            "sent_at"
        ]
    )
    .copy()
)


print("\n" + "=" * 90)
print("ELIGIBLE POPULATION")
print("=" * 90)

print(
    f"Customers observed overall   : "
    f"{df['customer_id'].nunique():,}"
)

print(
    f"First observed DPD <= {MAX_FIRST_DPD:<2}      : "
    f"{x['customer_id'].nunique():,}"
)

print(
    f"Population retained          : "
    f"{x['customer_id'].nunique() / df['customer_id'].nunique():.2%}"
)


# ============================================================
# 2. IDENTIFY DISCOUNT OFFER
# ============================================================

x["discount_offer_flag"] = (
    x["template"]
    .astype(str)
    .str.contains(
        "discount_offer",
        case=False,
        na=False
    )
)


# ============================================================
# 3. FIRST DISCOUNT OFFER PER CUSTOMER
# ============================================================

first_discount = (
    x.loc[
        x["discount_offer_flag"]
    ]
    .sort_values(
        [
            "customer_id",
            "sent_at"
        ]
    )
    .groupby("customer_id")
    .head(1)
    [
        [
            "customer_id",
            "sent_at",
            "outstanding_balance_brl"
        ]
    ]
    .rename(
        columns={
            "sent_at":
                "first_discount_at",

            "outstanding_balance_brl":
                "balance_at_first_discount_brl"
        }
    )
)


# ============================================================
# 4. PAYMENTS BEFORE FIRST DISCOUNT
# ============================================================
#
# This prevents us from comparing ALL historical recovery
# against only 85% of the remaining balance.
#
# Example:
#
# Initial balance          = 1,000
# Already paid             =   200
# Balance at discount      =   800
#
# Required recovery:
#
# 200 + 0.85 × 800 = 880
#
# ============================================================

discount_payment_base = (
    x.merge(
        first_discount[
            [
                "customer_id",
                "first_discount_at"
            ]
        ],
        on="customer_id",
        how="inner"
    )
)


discount_payment_base[
    "payment_before_discount"
] = np.where(

    discount_payment_base["sent_at"]
    <
    discount_payment_base["first_discount_at"],

    discount_payment_base[
        "amount_paid_brl"
    ],

    0
)


paid_before_discount = (
    discount_payment_base
    .groupby("customer_id")
    ["payment_before_discount"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "payment_before_discount":
                "paid_before_discount_brl"
        }
    )
)


# ============================================================
# 5. FULL CUSTOMER JOURNEY
# ============================================================

full_customer = (
    x.groupby("customer_id")
    .agg(

        total_messages=(
            "message_id",
            "count"
        ),

        total_recovered_brl=(
            "amount_paid_brl",
            "sum"
        ),

        initial_balance_brl=(
            "outstanding_balance_brl",
            "first"
        ),

        last_balance_brl=(
            "outstanding_balance_brl",
            "last"
        ),

        max_dpd=(
            "days_past_due",
            "max"
        ),

        first_message_at=(
            "sent_at",
            "min"
        ),

        last_message_at=(
            "sent_at",
            "max"
        ),

        received_discount_offer=(
            "discount_offer_flag",
            "max"
        )
    )
    .reset_index()
)


# ============================================================
# 6. ADD DISCOUNT INFORMATION
# ============================================================

full_customer = (
    full_customer
    .merge(
        first_discount,
        on="customer_id",
        how="left"
    )
    .merge(
        paid_before_discount,
        on="customer_id",
        how="left"
    )
)


full_customer[
    "paid_before_discount_brl"
] = (
    full_customer[
        "paid_before_discount_brl"
    ]
    .fillna(0)
)


# ============================================================
# 7. CANONICAL SETTLEMENT THRESHOLD
# ============================================================

full_customer[
    "settlement_threshold_brl"
] = np.where(

    full_customer[
        "received_discount_offer"
    ],

    (
        full_customer[
            "paid_before_discount_brl"
        ]
        +
        DISCOUNT_SETTLEMENT_RATE
        *
        full_customer[
            "balance_at_first_discount_brl"
        ]
    ),

    full_customer[
        "initial_balance_brl"
    ]
)


# ============================================================
# 8. FULL SETTLEMENT FLAG
# ============================================================

full_customer[
    "fully_settled"
] = (

    full_customer[
        "total_recovered_brl"
    ]

    >=

    (
        full_customer[
            "settlement_threshold_brl"
        ]
        -
        TOLERANCE_BRL
    )
)


# ============================================================
# 9. RECOVERY AGAINST SETTLEMENT REQUIREMENT
# ============================================================

full_customer[
    "recovery_vs_threshold_pct"
] = np.where(

    full_customer[
        "settlement_threshold_brl"
    ] > 0,

    (
        full_customer[
            "total_recovered_brl"
        ]
        /
        full_customer[
            "settlement_threshold_brl"
        ]
        * 100
    ),

    np.nan
)


# ============================================================
# 10. DPD FLAGS
# ============================================================

full_customer[
    "reached_dpd30"
] = (
    full_customer["max_dpd"] >= 30
)

full_customer[
    "reached_dpd60"
] = (
    full_customer["max_dpd"] >= 60
)


# ============================================================
# 11. OBSERVED JOURNEY LENGTH
# ============================================================

full_customer[
    "observed_journey_days"
] = (

    full_customer[
        "last_message_at"
    ]
    -
    full_customer[
        "first_message_at"
    ]

).dt.total_seconds() / 86400


# ============================================================
# 12. AUDIT CANONICAL FULL SETTLEMENT
# ============================================================

print("\n" + "=" * 90)
print("CANONICAL FULL SETTLEMENT AUDIT")
print("=" * 90)

print(
    f"Customers analysed           : "
    f"{len(full_customer):,}"
)

print(
    f"Fully settled                : "
    f"{full_customer['fully_settled'].sum():,}"
)

print(
    f"Fully settled rate           : "
    f"{full_customer['fully_settled'].mean():.2%}"
)

print(
    f"Received discount offer      : "
    f"{full_customer['received_discount_offer'].sum():,}"
)

print(
    f"Discount offer rate          : "
    f"{full_customer['received_discount_offer'].mean():.2%}"
)


# ============================================================
# 13. SETTLEMENT AUDIT BY DISCOUNT STATUS
# ============================================================

settlement_audit = (
    full_customer
    .groupby(
        "received_discount_offer",
        observed=True
    )
    .agg(

        customers=(
            "customer_id",
            "nunique"
        ),

        fully_settled=(
            "fully_settled",
            "sum"
        ),

        fully_settled_rate=(
            "fully_settled",
            "mean"
        ),

        avg_initial_balance_brl=(
            "initial_balance_brl",
            "mean"
        ),

        avg_paid_before_discount_brl=(
            "paid_before_discount_brl",
            "mean"
        ),

        avg_balance_at_discount_brl=(
            "balance_at_first_discount_brl",
            "mean"
        ),

        avg_settlement_threshold_brl=(
            "settlement_threshold_brl",
            "mean"
        ),

        avg_recovered_brl=(
            "total_recovered_brl",
            "mean"
        )
    )
    .reset_index()
)


settlement_audit[
    "fully_settled_rate"
] *= 100


print("\n" + "=" * 90)
print("SETTLEMENT BY DISCOUNT STATUS")
print("=" * 90)

display(
    settlement_audit.round(2)
)


# ============================================================
# 14. EARLY EXPOSURE — DPD 1–7
# ============================================================

early = (
    x.loc[
        x["days_past_due"].between(
            EARLY_DPD_MIN,
            EARLY_DPD_MAX
        )
    ]
    .copy()
)


early_customer = (
    early.groupby("customer_id")
    .agg(

        early_messages=(
            "message_id",
            "count"
        ),

        early_contact_days=(
            "sent_at",
            lambda s:
                s.dt.date.nunique()
        ),

        first_early_message_at=(
            "sent_at",
            "min"
        ),

        last_early_message_at=(
            "sent_at",
            "max"
        )
    )
    .reset_index()
)


# ============================================================
# 15. CADENCE BUCKETS
# ============================================================

early_customer[
    "early_cadence"
] = pd.cut(

    early_customer[
        "early_messages"
    ],

    bins=[
        0,
        2,
        4,
        np.inf
    ],

    labels=[
        "Low: 1-2",
        "Medium: 3-4",
        "High: 5+"
    ]
)


# ============================================================
# 16. POST EARLY-WINDOW OUTCOMES
# ============================================================
#
# Everything strictly after DPD 7.
#
# ============================================================

post = (
    x.loc[
        x["days_past_due"]
        >
        EARLY_DPD_MAX
    ]
    .copy()
)


post_customer = (
    post.groupby("customer_id")
    .agg(

        post_messages=(
            "message_id",
            "count"
        ),

        post_recovered_brl=(
            "amount_paid_brl",
            "sum"
        ),

        post_any_payment=(
            "amount_paid_brl",
            lambda s:
                (s.fillna(0) > 0).any()
        ),

        post_max_dpd=(
            "days_past_due",
            "max"
        )
    )
    .reset_index()
)


# ============================================================
# 17. COMBINE CUSTOMER LEVEL TABLES
# ============================================================

analysis = (
    early_customer

    .merge(
        post_customer,
        on="customer_id",
        how="left"
    )

    .merge(
        full_customer,
        on="customer_id",
        how="left"
    )
)


# ============================================================
# 18. CUSTOMERS WITH NO POST-DPD7 ACTIVITY
# ============================================================

analysis[
    "post_messages"
] = (
    analysis[
        "post_messages"
    ]
    .fillna(0)
    .astype(int)
)


analysis[
    "post_recovered_brl"
] = (
    analysis[
        "post_recovered_brl"
    ]
    .fillna(0)
)


analysis[
    "post_any_payment"
] = (
    analysis[
        "post_any_payment"
    ]
    .fillna(False)
    .astype(bool)
)


# ============================================================
# 19. SHARE OF MESSAGES CONCENTRATED EARLY
# ============================================================

analysis[
    "share_messages_early"
] = (

    analysis[
        "early_messages"
    ]

    /

    analysis[
        "total_messages"
    ]
)


# ============================================================
# 20. MAIN CADENCE SUMMARY
# ============================================================

cadence_summary = (
    analysis
    .groupby(
        "early_cadence",
        observed=True
    )
    .agg(

        # -----------------------------------------
        # Population
        # -----------------------------------------

        customers=(
            "customer_id",
            "nunique"
        ),

        # -----------------------------------------
        # Early pressure
        # -----------------------------------------

        avg_early_messages=(
            "early_messages",
            "mean"
        ),

        median_early_messages=(
            "early_messages",
            "median"
        ),

        avg_early_contact_days=(
            "early_contact_days",
            "mean"
        ),

        # -----------------------------------------
        # PRIMARY OUTCOME
        # -----------------------------------------

        fully_settled_rate=(
            "fully_settled",
            "mean"
        ),

        # -----------------------------------------
        # Payment after DPD7
        # -----------------------------------------

        post_payment_rate=(
            "post_any_payment",
            "mean"
        ),

        # -----------------------------------------
        # Messages after DPD7
        # -----------------------------------------

        avg_post_messages=(
            "post_messages",
            "mean"
        ),

        median_post_messages=(
            "post_messages",
            "median"
        ),

        # -----------------------------------------
        # Total messaging burden
        # -----------------------------------------

        avg_total_messages=(
            "total_messages",
            "mean"
        ),

        median_total_messages=(
            "total_messages",
            "median"
        ),

        # -----------------------------------------
        # Recovery
        # -----------------------------------------

        avg_recovered_brl=(
            "total_recovered_brl",
            "mean"
        ),

        median_recovered_brl=(
            "total_recovered_brl",
            "median"
        ),

        # -----------------------------------------
        # DPD
        # -----------------------------------------

        avg_max_dpd=(
            "max_dpd",
            "mean"
        ),

        reached_dpd30_rate=(
            "reached_dpd30",
            "mean"
        ),

        reached_dpd60_rate=(
            "reached_dpd60",
            "mean"
        ),

        # -----------------------------------------
        # Journey
        # -----------------------------------------

        avg_journey_days=(
            "observed_journey_days",
            "mean"
        )
    )
    .reset_index()
)


# ============================================================
# 21. CONVERT RATES TO %
# ============================================================

rate_columns = [

    "fully_settled_rate",
    "post_payment_rate",
    "reached_dpd30_rate",
    "reached_dpd60_rate"
]


for col in rate_columns:

    cadence_summary[col] = (
        cadence_summary[col]
        * 100
    )


# ============================================================
# 22. ROUND MAIN TABLE
# ============================================================

numeric_cols = (
    cadence_summary
    .select_dtypes(
        include="number"
    )
    .columns
)


cadence_summary[
    numeric_cols
] = (
    cadence_summary[
        numeric_cols
    ]
    .round(2)
)


print("\n" + "=" * 90)
print("EARLY CADENCE × COLLECTIONS OUTCOME")
print("=" * 90)

display(
    cadence_summary
)


# ============================================================
# 23. EXACT NUMBER OF EARLY MESSAGES
# ============================================================
#
# THIS IS THE MOST IMPORTANT TABLE.
#
# We don't want arbitrary buckets to hide a possible
# saturation point.
#
# ============================================================

exact_cadence = (
    analysis
    .groupby(
        "early_messages",
        observed=True
    )
    .agg(

        customers=(
            "customer_id",
            "nunique"
        ),

        fully_settled_rate=(
            "fully_settled",
            "mean"
        ),

        post_payment_rate=(
            "post_any_payment",
            "mean"
        ),

        avg_post_messages=(
            "post_messages",
            "mean"
        ),

        median_post_messages=(
            "post_messages",
            "median"
        ),

        avg_total_messages=(
            "total_messages",
            "mean"
        ),

        median_total_messages=(
            "total_messages",
            "median"
        ),

        avg_recovered_brl=(
            "total_recovered_brl",
            "mean"
        ),

        median_recovered_brl=(
            "total_recovered_brl",
            "median"
        ),

        avg_max_dpd=(
            "max_dpd",
            "mean"
        ),

        reached_dpd30_rate=(
            "reached_dpd30",
            "mean"
        ),

        reached_dpd60_rate=(
            "reached_dpd60",
            "mean"
        ),

        avg_journey_days=(
            "observed_journey_days",
            "mean"
        )
    )
    .reset_index()
)


for col in [

    "fully_settled_rate",
    "post_payment_rate",
    "reached_dpd30_rate",
    "reached_dpd60_rate"

]:

    exact_cadence[col] = (
        exact_cadence[col]
        * 100
    )


numeric_cols = (
    exact_cadence
    .select_dtypes(
        include="number"
    )
    .columns
)


exact_cadence[
    numeric_cols
] = (
    exact_cadence[
        numeric_cols
    ]
    .round(2)
)


print("\n" + "=" * 90)
print("EXACT NUMBER OF EARLY MESSAGES")
print("=" * 90)

display(
    exact_cadence
)


# ============================================================
# 24. SETTLEMENT-FOCUSED VIEW
# ============================================================
#
# Compact table specifically for the business question:
#
# Are more early messages associated with:
#
#   settlement ↑
#   later messages ↓
#   total messages ↓
#
# ============================================================

settlement_cadence = (
    exact_cadence[
        [
            "early_messages",
            "customers",
            "fully_settled_rate",
            "post_payment_rate",
            "avg_post_messages",
            "avg_total_messages",
            "avg_recovered_brl",
            "avg_max_dpd",
            "reached_dpd30_rate",
            "reached_dpd60_rate"
        ]
    ]
    .copy()
)


print("\n" + "=" * 90)
print("CADENCE × SETTLEMENT — BUSINESS VIEW")
print("=" * 90)

display(
    settlement_cadence
)


# ============================================================
# 25. SETTLED vs NOT SETTLED
# ============================================================
#
# Additional diagnostic:
#
# How different were the journeys of customers who eventually
# settled versus those who did not?
#
# ============================================================

settlement_profile = (
    analysis
    .groupby(
        "fully_settled",
        observed=True
    )
    .agg(

        customers=(
            "customer_id",
            "nunique"
        ),

        avg_early_messages=(
            "early_messages",
            "mean"
        ),

        median_early_messages=(
            "early_messages",
            "median"
        ),

        avg_post_messages=(
            "post_messages",
            "mean"
        ),

        avg_total_messages=(
            "total_messages",
            "mean"
        ),

        median_total_messages=(
            "total_messages",
            "median"
        ),

        avg_max_dpd=(
            "max_dpd",
            "mean"
        ),

        avg_journey_days=(
            "observed_journey_days",
            "mean"
        ),

        avg_recovered_brl=(
            "total_recovered_brl",
            "mean"
        )
    )
    .reset_index()
)


numeric_cols = (
    settlement_profile
    .select_dtypes(
        include="number"
    )
    .columns
)


settlement_profile[
    numeric_cols
] = (
    settlement_profile[
        numeric_cols
    ]
    .round(2)
)


print("\n" + "=" * 90)
print("SETTLED vs NOT SETTLED — JOURNEY PROFILE")
print("=" * 90)

display(
    settlement_profile
)


# ============================================================
# 26. CUSTOMER-LEVEL ANALYTICAL BASE
# ============================================================

print("\n" + "=" * 90)
print("CUSTOMER-LEVEL ANALYTICAL BASE")
print("=" * 90)

print(
    f"Customers analysed            : "
    f"{len(analysis):,}"
)

print(
    f"Fully settled                 : "
    f"{analysis['fully_settled'].sum():,}"
)

print(
    f"Fully settled rate            : "
    f"{analysis['fully_settled'].mean():.2%}"
)


display(

    analysis[
        [
            "customer_id",

            "early_cadence",
            "early_messages",
            "early_contact_days",

            "received_discount_offer",

            "initial_balance_brl",
            "paid_before_discount_brl",
            "balance_at_first_discount_brl",
            "settlement_threshold_brl",

            "total_recovered_brl",
            "recovery_vs_threshold_pct",

            "fully_settled",

            "post_any_payment",
            "post_messages",
            "total_messages",

            "max_dpd",
            "reached_dpd30",
            "reached_dpd60",

            "observed_journey_days"
        ]
    ]
    .head(30)
)


# ============================================================
# 27. FINAL SANITY CHECKS
# ============================================================

print("\n" + "=" * 90)
print("FINAL SANITY CHECKS")
print("=" * 90)


print(
    "Missing settlement threshold :",
    full_customer[
        "settlement_threshold_brl"
    ].isna().sum()
)


print(
    "Missing fully_settled        :",
    full_customer[
        "fully_settled"
    ].isna().sum()
)


print(
    "Threshold <= 0               :",
    (
        full_customer[
            "settlement_threshold_brl"
        ] <= 0
    ).sum()
)


print(
    "Recovery < 0                 :",
    (
        full_customer[
            "total_recovered_brl"
        ] < 0
    ).sum()
)


print("\nTEST 1 COMPLETE.")

TEST 1 — EARLY CADENCE vs EVENTUAL FULL SETTLEMENT
Rows                         : 75,406
Customers                    : 11,724
Observed period              : 2026-06-01 → 2026-08-31

ELIGIBLE POPULATION
Customers observed overall   : 11,724
First observed DPD <= 3       : 8,068
Population retained          : 68.82%

CANONICAL FULL SETTLEMENT AUDIT
Customers analysed           : 8,068
Fully settled                : 2,591
Fully settled rate           : 32.11%
Received discount offer      : 2,432
Discount offer rate          : 30.14%

SETTLEMENT BY DISCOUNT STATUS


,received_discount_offer,customers,fully_settled,fully_settled_rate,avg_initial_balance_brl,avg_paid_before_discount_brl,avg_balance_at_discount_brl,avg_settlement_threshold_brl,avg_recovered_brl
0,False,5636,2291,40.65,849.13,0.00,NaN,849.13,371.86
1,True,2432,300,12.34,837.07,68.21,768.87,721.74,153.04



EARLY CADENCE × COLLECTIONS OUTCOME


,early_cadence,customers,avg_early_messages,median_early_messages,avg_early_contact_days,fully_settled_rate,post_payment_rate,avg_post_messages,median_post_messages,avg_total_messages,median_total_messages,avg_recovered_brl,median_recovered_brl,avg_max_dpd,reached_dpd30_rate,reached_dpd60_rate,avg_journey_days
0,Low: 1-2,4710,1.60,2.00,1.60,36.39,24.63,3.80,3.00,5.40,5.00,333.69,0.00,23.68,36.58,1.59,21.84
1,Medium: 3-4,3093,3.29,3.00,3.29,26.58,27.51,4.89,5.00,8.18,8.00,272.13,0.00,30.13,47.79,2.52,28.50
2,High: 5+,265,5.11,5.00,5.11,20.75,25.66,5.04,5.00,10.15,10.00,206.13,0.00,31.47,51.32,2.64,30.09



EXACT NUMBER OF EARLY MESSAGES


...


SETTLED vs NOT SETTLED — JOURNEY PROFILE


,fully_settled,customers,avg_early_messages,median_early_messages,avg_post_messages,avg_total_messages,median_total_messages,avg_max_dpd,avg_journey_days,avg_recovered_brl
0,False,5477,2.47,2.00,5.25,7.72,8.00,32.14,30.38,70.53
1,True,2591,2.15,2.00,2.14,4.29,4.00,14.30,12.58,803.42



CUSTOMER-LEVEL ANALYTICAL BASE
Customers analysed            : 8,068
Fully settled                 : 2,591
Fully settled rate            : 32.11%


,customer_id,early_cadence,early_messages,early_contact_days,received_discount_offer,initial_balance_brl,paid_before_discount_brl,balance_at_first_discount_brl,settlement_threshold_brl,total_recovered_brl,recovery_vs_threshold_pct,fully_settled,post_any_payment,post_messages,total_messages,max_dpd,reached_dpd30,reached_dpd60,observed_journey_days
0,C000001,Medium: 3-4,3,3,False,934.58,0.00,NaN,934.58,934.58,100.00,True,True,4,7,26,False,False,22.82
1,C000003,Medium: 3-4,4,4,False,758.22,0.00,NaN,758.22,0.00,0.00,False,False,8,12,60,True,True,57.04
2,C000004,Medium: 3-4,3,3,False,"1,331.09",0.00,NaN,"1,331.09","1,331.09",100.00,True,True,2,5,11,False,False,9.72
3,C000005,Medium: 3-4,3,3,False,250.35,0.00,NaN,250.35,0.00,0.00,False,False,2,5,21,False,False,20.19
4,C000006,Medium: 3-4,4,4,True,488.23,0.00,488.23,415.00,415.00,100.00,True,True,7,11,34,True,False,32.02
5,C000007,Low: 1-2,2,2,False,314.44,0.00,NaN,314.44,0.00,0.00,False,False,9,11,49,True,False,48.05
6,C000008,Medium: 3-4,4,4,False,927.60,0.00,NaN,927.60,0.00,0.00,False,False,8,12,53,True,False,51.71
7,C000009,Medium: 3-4,3,3,True,878.15,0.00,878.15,746.43,878.15,117.65,True,True,4,7,56,True,False,54.81
8,C000011,Low: 1-2,1,1,True,966.15,456.62,509.53,889.72,456.62,51.32,False,True,10,11,48,True,False,46.96
9,C000012,Medium: 3-4,3,3,False,320.23,0.00,NaN,320.23,320.23,100.00,True,True,1,4,18,False,False,16.67



FINAL SANITY CHECKS
Missing settlement threshold : 0
Missing fully_settled        : 0
Threshold <= 0               : 0
Recovery < 0                 : 0

TEST 1 COMPLETE.


In [19]:
# ============================================================
# TEST 2 — MESSAGE GAP / CADENCE TIMING
# ============================================================
#
# BUSINESS QUESTION
#
# Given that a customer is still in the collections journey
# and receives another attempt:
#
# Does contacting them sooner vs later appear associated with:
#
#   - payment response?
#   - eventual full settlement?
#   - fewer subsequent messages?
#   - shorter collections journey?
#
#
# UNIT OF ANALYSIS:
#
# Customer × message transition
#
# Examples:
#
#   attempt 1 -> attempt 2
#   attempt 2 -> attempt 3
#   attempt 3 -> attempt 4
#   attempt 4 -> attempt 5
#
#
# IMPORTANT:
#
# This is still observational.
# It does NOT establish causal effect.
#
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# PARAMETERS
# ============================================================

MAX_TRANSITION = 5

# We focus primarily on the early collections journey.
MAX_DPD_AT_PREVIOUS_ATTEMPT = 30


# ============================================================
# 1. PREPARE MESSAGE-LEVEL JOURNEY
# ============================================================

msg = (
    x.sort_values(
        ["customer_id", "sent_at"]
    )
    .copy()
)


# Sequential attempt number
msg["attempt_number"] = (
    msg.groupby("customer_id")
    .cumcount()
    + 1
)


# Previous message timestamp
msg["previous_sent_at"] = (
    msg.groupby("customer_id")
    ["sent_at"]
    .shift(1)
)


# Previous DPD
msg["previous_dpd"] = (
    msg.groupby("customer_id")
    ["days_past_due"]
    .shift(1)
)


# Previous template
msg["previous_template"] = (
    msg.groupby("customer_id")
    ["template"]
    .shift(1)
)


# ============================================================
# 2. GAP BETWEEN CONSECUTIVE MESSAGES
# ============================================================

msg["gap_hours"] = (
    (
        msg["sent_at"]
        -
        msg["previous_sent_at"]
    )
    .dt.total_seconds()
    / 3600
)


msg["gap_days"] = (
    msg["gap_hours"]
    / 24
)


# ============================================================
# 3. TRANSITION NUMBER
# ============================================================
#
# Current attempt = 2
# means transition:
#
#     attempt 1 -> attempt 2
#
# ============================================================

msg["transition_number"] = (
    msg["attempt_number"]
    - 1
)


msg["transition"] = (
    msg["transition_number"]
    .astype("Int64")
    .astype(str)
    +
    " → "
    +
    msg["attempt_number"]
    .astype("Int64")
    .astype(str)
)


# ============================================================
# 4. KEEP VALID TRANSITIONS
# ============================================================

transitions = (
    msg.loc[
        msg["attempt_number"].between(
            2,
            MAX_TRANSITION + 1
        )
        &
        msg["previous_dpd"].le(
            MAX_DPD_AT_PREVIOUS_ATTEMPT
        )
        &
        msg["gap_days"].notna()
        &
        msg["gap_days"].ge(0)
    ]
    .copy()
)


print("=" * 95)
print("TEST 2 — MESSAGE GAP / CADENCE TIMING")
print("=" * 95)

print(
    f"Customers                     : "
    f"{transitions['customer_id'].nunique():,}"
)

print(
    f"Transitions                   : "
    f"{len(transitions):,}"
)

print(
    f"Median gap                    : "
    f"{transitions['gap_days'].median():.2f} days"
)

print(
    f"Mean gap                      : "
    f"{transitions['gap_days'].mean():.2f} days"
)


# ============================================================
# 5. GAP BUCKET
# ============================================================
#
# Operational cadence groups
#
# ============================================================

transitions["gap_bucket"] = pd.cut(

    transitions["gap_days"],

    bins=[
        -0.001,
        1,
        2,
        3,
        5,
        np.inf
    ],

    labels=[
        "≤1 day",
        "1-2 days",
        "2-3 days",
        "3-5 days",
        "5+ days"
    ],

    include_lowest=True,
    right=True
)


# ============================================================
# 6. ADD EVENTUAL CUSTOMER OUTCOMES
# ============================================================

customer_outcomes = (
    full_customer[
        [
            "customer_id",
            "fully_settled",
            "total_messages",
            "total_recovered_brl",
            "max_dpd",
            "reached_dpd30",
            "reached_dpd60",
            "observed_journey_days"
        ]
    ]
    .copy()
)


transitions = (
    transitions
    .merge(
        customer_outcomes,
        on="customer_id",
        how="left"
    )
)


# ============================================================
# 7. PAYMENT RESPONSE TO CURRENT MESSAGE
# ============================================================
#
# amount_paid_brl belongs to the current send's
# attributed 72h response window.
#
# Therefore:
#
# payment_response = payment attributed after current attempt.
#
# ============================================================

transitions["payment_response"] = (
    transitions["amount_paid_brl"] > 0
)


# ============================================================
# 8. NUMBER OF MESSAGES AFTER CURRENT ATTEMPT
# ============================================================

transitions[
    "messages_after_current"
] = (

    transitions["total_messages"]
    -
    transitions["attempt_number"]
).clip(lower=0)


# ============================================================
# 9. PRIMARY SUMMARY:
#    GAP × TRANSITION NUMBER
# ============================================================

gap_transition_summary = (
    transitions
    .groupby(
        [
            "transition",
            "gap_bucket"
        ],
        observed=True
    )
    .agg(

        customers=(
            "customer_id",
            "nunique"
        ),

        avg_gap_days=(
            "gap_days",
            "mean"
        ),

        median_gap_days=(
            "gap_days",
            "median"
        ),

        payment_response_rate=(
            "payment_response",
            "mean"
        ),

        fully_settled_rate=(
            "fully_settled",
            "mean"
        ),

        avg_messages_after_current=(
            "messages_after_current",
            "mean"
        ),

        median_messages_after_current=(
            "messages_after_current",
            "median"
        ),

        avg_total_messages=(
            "total_messages",
            "mean"
        ),

        avg_recovered_brl=(
            "total_recovered_brl",
            "mean"
        ),

        avg_max_dpd=(
            "max_dpd",
            "mean"
        ),

        reached_dpd30_rate=(
            "reached_dpd30",
            "mean"
        ),

        reached_dpd60_rate=(
            "reached_dpd60",
            "mean"
        )
    )
    .reset_index()
)


# Convert rates to %
for col in [
    "payment_response_rate",
    "fully_settled_rate",
    "reached_dpd30_rate",
    "reached_dpd60_rate"
]:

    gap_transition_summary[col] *= 100


numeric_cols = (
    gap_transition_summary
    .select_dtypes(include="number")
    .columns
)

gap_transition_summary[
    numeric_cols
] = (
    gap_transition_summary[
        numeric_cols
    ]
    .round(2)
)


print("\n" + "=" * 95)
print("GAP × ATTEMPT TRANSITION")
print("=" * 95)

display(
    gap_transition_summary
)


# ============================================================
# 10. TRANSITION 1 -> 2
# ============================================================

transition_1_2 = (
    gap_transition_summary.loc[
        gap_transition_summary[
            "transition"
        ] == "1 → 2"
    ]
    .copy()
)


print("\n" + "=" * 95)
print("ATTEMPT 1 → 2")
print("=" * 95)

display(
    transition_1_2[
        [
            "gap_bucket",
            "customers",
            "avg_gap_days",
            "payment_response_rate",
            "fully_settled_rate",
            "avg_messages_after_current",
            "avg_total_messages",
            "avg_recovered_brl",
            "reached_dpd30_rate"
        ]
    ]
)


# ============================================================
# 11. TRANSITION 2 -> 3
# ============================================================

transition_2_3 = (
    gap_transition_summary.loc[
        gap_transition_summary[
            "transition"
        ] == "2 → 3"
    ]
    .copy()
)


print("\n" + "=" * 95)
print("ATTEMPT 2 → 3")
print("=" * 95)

display(
    transition_2_3[
        [
            "gap_bucket",
            "customers",
            "avg_gap_days",
            "payment_response_rate",
            "fully_settled_rate",
            "avg_messages_after_current",
            "avg_total_messages",
            "avg_recovered_brl",
            "reached_dpd30_rate"
        ]
    ]
)


# ============================================================
# 12. TRANSITION 3 -> 4
# ============================================================

transition_3_4 = (
    gap_transition_summary.loc[
        gap_transition_summary[
            "transition"
        ] == "3 → 4"
    ]
    .copy()
)


print("\n" + "=" * 95)
print("ATTEMPT 3 → 4")
print("=" * 95)

display(
    transition_3_4[
        [
            "gap_bucket",
            "customers",
            "avg_gap_days",
            "payment_response_rate",
            "fully_settled_rate",
            "avg_messages_after_current",
            "avg_total_messages",
            "avg_recovered_brl",
            "reached_dpd30_rate"
        ]
    ]
)


# ============================================================
# 13. TRANSITION 4 -> 5
# ============================================================

transition_4_5 = (
    gap_transition_summary.loc[
        gap_transition_summary[
            "transition"
        ] == "4 → 5"
    ]
    .copy()
)


print("\n" + "=" * 95)
print("ATTEMPT 4 → 5")
print("=" * 95)

display(
    transition_4_5[
        [
            "gap_bucket",
            "customers",
            "avg_gap_days",
            "payment_response_rate",
            "fully_settled_rate",
            "avg_messages_after_current",
            "avg_total_messages",
            "avg_recovered_brl",
            "reached_dpd30_rate"
        ]
    ]
)


# ============================================================
# 14. TRANSITION 5 -> 6
# ============================================================

transition_5_6 = (
    gap_transition_summary.loc[
        gap_transition_summary[
            "transition"
        ] == "5 → 6"
    ]
    .copy()
)


print("\n" + "=" * 95)
print("ATTEMPT 5 → 6")
print("=" * 95)

display(
    transition_5_6[
        [
            "gap_bucket",
            "customers",
            "avg_gap_days",
            "payment_response_rate",
            "fully_settled_rate",
            "avg_messages_after_current",
            "avg_total_messages",
            "avg_recovered_brl",
            "reached_dpd30_rate"
        ]
    ]
)


# ============================================================
# 15. OVERALL GAP VIEW
# ============================================================
#
# Descriptive only.
#
# The transition-specific tables above are more important.
#
# ============================================================

overall_gap = (
    transitions
    .groupby(
        "gap_bucket",
        observed=True
    )
    .agg(

        transitions=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        avg_gap_days=(
            "gap_days",
            "mean"
        ),

        payment_response_rate=(
            "payment_response",
            "mean"
        ),

        fully_settled_rate=(
            "fully_settled",
            "mean"
        ),

        avg_messages_after_current=(
            "messages_after_current",
            "mean"
        ),

        avg_total_messages=(
            "total_messages",
            "mean"
        ),

        avg_recovered_brl=(
            "total_recovered_brl",
            "mean"
        ),

        reached_dpd30_rate=(
            "reached_dpd30",
            "mean"
        )
    )
    .reset_index()
)


for col in [
    "payment_response_rate",
    "fully_settled_rate",
    "reached_dpd30_rate"
]:

    overall_gap[col] *= 100


numeric_cols = (
    overall_gap
    .select_dtypes(include="number")
    .columns
)

overall_gap[
    numeric_cols
] = (
    overall_gap[
        numeric_cols
    ]
    .round(2)
)


print("\n" + "=" * 95)
print("OVERALL GAP VIEW — DESCRIPTIVE")
print("=" * 95)

display(
    overall_gap
)


# ============================================================
# 16. SAMPLE SIZE / RELIABILITY AUDIT
# ============================================================
#
# Flag cells with very small populations.
#
# ============================================================

reliability = (
    gap_transition_summary[
        [
            "transition",
            "gap_bucket",
            "customers",
            "payment_response_rate",
            "fully_settled_rate"
        ]
    ]
    .copy()
)


reliability["sample_flag"] = np.select(

    [
        reliability["customers"] < 30,
        reliability["customers"] < 100
    ],

    [
        "VERY SMALL (<30)",
        "SMALL (30-99)"
    ],

    default="OK (100+)"
)


print("\n" + "=" * 95)
print("SAMPLE SIZE AUDIT")
print("=" * 95)

display(
    reliability
)


# ============================================================
# 17. GAP DISTRIBUTION BY TRANSITION
# ============================================================

gap_distribution = (
    transitions
    .groupby(
        [
            "transition",
            "gap_bucket"
        ],
        observed=True
    )
    .size()
    .reset_index(
        name="transitions"
    )
)


gap_distribution[
    "share_within_transition"
] = (

    gap_distribution[
        "transitions"
    ]

    /

    gap_distribution
    .groupby("transition")
    ["transitions"]
    .transform("sum")

    * 100
)


gap_distribution[
    "share_within_transition"
] = (
    gap_distribution[
        "share_within_transition"
    ]
    .round(2)
)


print("\n" + "=" * 95)
print("CURRENT CADENCE DISTRIBUTION")
print("=" * 95)

display(
    gap_distribution
)


print("\nTEST 2 COMPLETE.")

TEST 2 — MESSAGE GAP / CADENCE TIMING
Customers                     : 7,252
Transitions                   : 29,443
Median gap                    : 2.69 days
Mean gap                      : 3.67 days

GAP × ATTEMPT TRANSITION


,transition,gap_bucket,customers,avg_gap_days,median_gap_days,payment_response_rate,fully_settled_rate,avg_messages_after_current,median_messages_after_current,avg_total_messages,avg_recovered_brl,avg_max_dpd,reached_dpd30_rate,reached_dpd60_rate
0,1 → 2,≤1 day,1216,0.84,0.87,8.14,27.14,5.73,6.00,7.73,261.35,28.25,44.33,2.38
1,1 → 2,1-2 days,1847,1.41,1.27,10.12,31.02,5.56,5.00,7.56,307.40,28.22,44.94,1.89
2,1 → 2,2-3 days,1212,2.44,2.31,7.34,26.57,5.51,5.00,7.51,251.74,29.02,46.29,2.39
3,1 → 2,3-5 days,1602,3.86,3.91,9.30,28.34,5.06,5.00,7.06,288.83,29.40,45.76,2.43
4,1 → 2,5+ days,1375,7.64,6.89,8.95,27.42,4.43,4.00,6.43,276.05,31.20,49.02,2.04
5,2 → 3,≤1 day,1040,0.84,0.87,8.56,26.35,5.33,5.00,8.33,267.27,31.18,50.67,2.40
6,2 → 3,1-2 days,1673,1.42,1.28,8.67,25.46,5.20,5.00,8.20,266.60,30.63,48.77,2.33
7,2 → 3,2-3 days,1090,2.46,2.30,8.17,25.41,4.93,5.00,7.93,250.28,30.92,49.36,2.48
8,2 → 3,3-5 days,1438,3.89,3.94,9.25,25.80,4.63,4.00,7.63,268.42,31.60,50.07,2.71
9,2 → 3,5+ days,1337,7.85,6.87,8.75,24.23,3.92,4.00,6.92,244.34,33.87,54.97,2.24



ATTEMPT 1 → 2


,gap_bucket,customers,avg_gap_days,payment_response_rate,fully_settled_rate,avg_messages_after_current,avg_total_messages,avg_recovered_brl,reached_dpd30_rate
0,≤1 day,1216,0.84,8.14,27.14,5.73,7.73,261.35,44.33
1,1-2 days,1847,1.41,10.12,31.02,5.56,7.56,307.40,44.94
2,2-3 days,1212,2.44,7.34,26.57,5.51,7.51,251.74,46.29
3,3-5 days,1602,3.86,9.30,28.34,5.06,7.06,288.83,45.76
4,5+ days,1375,7.64,8.95,27.42,4.43,6.43,276.05,49.02



ATTEMPT 2 → 3


...


SAMPLE SIZE AUDIT


,transition,gap_bucket,customers,payment_response_rate,fully_settled_rate,sample_flag
0,1 → 2,≤1 day,1216,8.14,27.14,OK (100+)
1,1 → 2,1-2 days,1847,10.12,31.02,OK (100+)
2,1 → 2,2-3 days,1212,7.34,26.57,OK (100+)
3,1 → 2,3-5 days,1602,9.30,28.34,OK (100+)
4,1 → 2,5+ days,1375,8.95,27.42,OK (100+)
5,2 → 3,≤1 day,1040,8.56,26.35,OK (100+)
6,2 → 3,1-2 days,1673,8.67,25.46,OK (100+)
7,2 → 3,2-3 days,1090,8.17,25.41,OK (100+)
8,2 → 3,3-5 days,1438,9.25,25.80,OK (100+)
9,2 → 3,5+ days,1337,8.75,24.23,OK (100+)



CURRENT CADENCE DISTRIBUTION


,transition,gap_bucket,transitions,share_within_transition
0,1 → 2,≤1 day,1216,16.77
1,1 → 2,1-2 days,1847,25.47
2,1 → 2,2-3 days,1212,16.71
3,1 → 2,3-5 days,1602,22.09
4,1 → 2,5+ days,1375,18.96
5,2 → 3,≤1 day,1040,15.81
6,2 → 3,1-2 days,1673,25.43
7,2 → 3,2-3 days,1090,16.57
8,2 → 3,3-5 days,1438,21.86
9,2 → 3,5+ days,1337,20.33



TEST 2 COMPLETE.
